# Social media use during Covid-19 pandemic

## 1 Introduction

In the present notebook, we will use a parametric and a non-parametric test to assess the difference in the amounts of time spent on social media between groups of students who used either a computer or a smartphone for their coursework in the Covid-19 pandemic.

## 2 Executive summary

[SUMMARY HERE]

## 3 Report

### 3.1 Report introduction

This is an interesting cross-section of data from a survey of 1078 respondents. We have individuals' IDs, ages and the region of residence. We further have information on on the device used to access online classes and on the prefered social media platform and stress activity. We further have individuals' time spent on classes, self-study, fitness, sleep, social media and TV. Last but not least, we have responses on the daily number of meals, change in weight, presence of health issues and whether the respondents feel connected to their family and friends.

The present dataset comes from DataCamp's [A/B Testing in R](https://app.datacamp.com/learn/courses/ab-testing-in-r?cf-exp=comm--ai-native-cdp-layout%3Aon) course.

### 3.2 Deliverables

[DELIVERABLES HERE]

### 3.3 Methods

[METHODS HERE]

### 3.4 Discussion

[DISCUSSION HERE]

### 3.5 Recommendations

[RECOMMENDATIONS HERE]

In [3]:
# Libraries
library(tidyverse)
library(readxl)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Workings

### Helpers

In [10]:
# Custom functionality and libraries
#devtools::install_github("tros01/google_advanced_data_analytics", subdir = "custom_libraries/utils")
library(gada.utils)


#### Graphing setup

In [1]:
# Set up Jupyter graph display
options(
  repr.plot.width = 10,
  repr.plot.height = 6,
  repr.plot.res = 150
)

### Load and verify the data

In [44]:
# Load up the data into a df
df <- read_csv(
    file.path(r"(data/covid.csv)"),
    col_names = TRUE,
    col_types = NULL,
    show_col_types = FALSE
)

In [9]:
# Dimension check
df |> dim()

[1] 1078   21

The dataset appears to be complete.

In [45]:
# Format column names
df <- df |> janitor::clean_names()

In [7]:
# Display a sample
df |> sample_n(5)

x_1,x,id,region_of_residence,age_of_subject,time_spent_on_online_class,rating_of_online_class_experience,medium_for_online_class,time_spent_on_self_study,time_spent_on_fitness,⋯,time_spent_on_social_media,prefered_social_media_platform,time_spent_on_tv,number_of_meals_per_day,change_in_your_weight,health_issue_during_lockdown,stress_busters,time_utilized,do_you_find_yourself_more_connected_with_your_family_close_friends_relatives,what_you_miss_the_most
<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,⋯,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
940,940,1033,Outside Delhi-NCR,25,0,Very poor,Smartphone,2,0.0,⋯,5,Facebook,0,3,Remain Constant,NO,Listening to music,NO,YES,Roaming around freely
40,40,R44,Delhi-NCR,21,5,Very poor,Laptop/Desktop,3,1.0,⋯,1,Instagram,0,2,Increased,NO,Listening to music,NO,NO,Eating outside
984,984,1081,Delhi-NCR,15,3,Average,Smartphone,4,0.5,⋯,1,Instagram,1.5,4,Increased,NO,Listening to music,YES,YES,Eating outside
35,35,R38,Outside Delhi-NCR,20,4,Average,Laptop/Desktop,2,0.0,⋯,2,Instagram,0,3,Increased,NO,Sleep,NO,YES,Eating outside
582,582,R657,Delhi-NCR,19,4,Very poor,Smartphone,5,0.0,⋯,2,Whatsapp,0,3,Increased,NO,Reading books,YES,YES,School/college


In [11]:
# df structure
df |> str()

spc_tbl_ [1,078 × 21] (S3: spec_tbl_df/tbl_df/tbl/data.frame)
 $ x_1                                                                         : num [1:1078] 2 4 6 7 11 12 13 14 15 16 ...
 $ x                                                                           : num [1:1078] 2 4 6 7 11 12 13 14 15 16 ...
 $ id                                                                          : chr [1:1078] "R2" "R4" "R6" "R7" ...
 $ region_of_residence                                                         : chr [1:1078] "Delhi-NCR" "Delhi-NCR" "Delhi-NCR" "Delhi-NCR" ...
 $ age_of_subject                                                              : num [1:1078] 21 20 21 19 21 21 22 20 22 20 ...
 $ time_spent_on_online_class                                                  : num [1:1078] 0 3 0 2 1 3 1 5 3 0 ...
 $ rating_of_online_class_experience                                           : chr [1:1078] "Excellent" "Very poor" "Very poor" "Very poor" ...
 $ medium_for_online_class        

- region of residence, class rating, medium, change in weight and binary variables are factors
- prefered social media platform is potentially a factor
- time spent on TV is a numeric variable

In [ ]:
# Unordered factor variables
factor_vars <- c(
    "region_of_residence",    
    "medium_for_online_class",
    "prefered_social_media_platform",
    "change_in_your_weight",
    "health_issue_during_lockdown",
    "time_utilized",
    "do_you_find_yourself_more_connected_with_your_family_close_friends_relatives"
)

In [47]:
# Unordered factors
for (var in factor_vars) {
    levels <- df |>
        select(all_of(var)) |>
        drop_na() |> 
        pull() |>
        str_squish() |> 
        str_to_lower() |>
        str_replace_all("[ -/]", "_") |>
        unique()

    df <- df |>
        mutate(
            !!sym(var) := str_replace_all(str_to_lower(str_squish(!!sym(var))), "[ -/]", "_"),
            !!sym(var) := factor(!!sym(var), levels = levels, ordered = FALSE)
        )
}

In [48]:
# Ordered factors and numeric vars
levels <- df |>
        select(rating_of_online_class_experience) |>
        drop_na() |> 
        pull() |>
        str_squish() |> 
        str_to_lower() |>
        str_replace_all("[ -/]", "_") |>
        unique()

ordered_levels <- levels[c(2, 5, 3, 4, 1)]

df <- df |>
    mutate(
        rating_of_online_class_experience = factor(rating_of_online_class_experience, levels = ordered_levels, ordered = TRUE),
        time_spent_on_tv = as.numeric(time_spent_on_tv)
    )

Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `time_spent_on_tv = as.numeric(time_spent_on_tv)`.
Caused by warning:
! NAs introduced by coercion"


In [50]:
# Sample the df
df |> sample_n(5)

x_1,x,id,region_of_residence,age_of_subject,time_spent_on_online_class,rating_of_online_class_experience,medium_for_online_class,time_spent_on_self_study,time_spent_on_fitness,⋯,time_spent_on_social_media,prefered_social_media_platform,time_spent_on_tv,number_of_meals_per_day,change_in_your_weight,health_issue_during_lockdown,stress_busters,time_utilized,do_you_find_yourself_more_connected_with_your_family_close_friends_relatives,what_you_miss_the_most
<dbl>,<dbl>,<chr>,<fct>,<dbl>,<dbl>,<ord>,<fct>,<dbl>,<dbl>,⋯,<dbl>,<fct>,<dbl>,<dbl>,<fct>,<fct>,<chr>,<fct>,<fct>,<chr>
285,285,R315,delhi_ncr,19,1,NA,laptop_desktop,3,2,⋯,1,instagram,2,2,remain_constant,yes,Online gaming,no,no,School/college
154,154,R165,delhi_ncr,22,1,NA,smartphone,4,1,⋯,2,linkedin,0,2,increased,yes,Watching web series,no,no,Roaming around freely
995,995,1093,delhi_ncr,17,6,NA,smartphone,1,0,⋯,1,reddit,0,2,remain_constant,no,Online gaming,no,no,Metro
694,694,R774,delhi_ncr,40,6,NA,laptop_desktop,3,0,⋯,2,whatsapp,2,3,remain_constant,no,No able to reduce the stress,no,yes,Travelling
610,610,R686,delhi_ncr,21,3,NA,laptop_desktop,4,1,⋯,1,youtube,2,4,decreased,no,Meditation,no,yes,"Friends , relatives"


#### Investigate NAs, negatives, duplicates